<a href="https://colab.research.google.com/github/iav2002/Assignment_Advanced_Topics_In_DeepLearning/blob/main/part2_experiment_2(LoRA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 2 — LoRA Rank Study on ResNet50

Investigates how the LoRA rank affects accuracy and trainable parameter count. We apply LoRA to the 9 convolutions of `layer4` (the most abstract block), keep the rest of the backbone frozen, and train the head normally. Four ranks are compared: 4, 8, 16, 32.

We use HuggingFace's `peft` library rather than implementing LoRA from scratch. A manual implementation was prototyped but `peft` handles the wrapping, frozen-weight bookkeeping, and parameter counting cleanly enough that the manual version added complexity without adding insight. The implementation logic is identical underneath: each wrapped conv gets two small matrices A and B, the original weight stays frozen, the forward pass computes `base(x) + scaling * B(A(x))` with B initialized to zero so training starts from the pretrained model exactly.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json, random, time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
if device.type == 'cuda':
    print(f'gpu: {torch.cuda.get_device_name(0)}')

Mounted at /content/drive
device: cuda
gpu: NVIDIA L4


In [2]:
# install peft - not in colab by default
!pip install -q peft
import peft
print(f'peft version: {peft.__version__}')

peft version: 0.19.1


## 2. Paths and shared training config

Same config as Exp 1 for direct comparability. Ranks defined here so the experiment loop reads from one place.

In [3]:
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdvancedDL')
INDEX_DIR = DRIVE_ROOT / 'sample_index'
RESULTS_DIR = DRIVE_ROOT / 'results' / 'exp2_lora'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# same training config as exp1 - keeps comparison clean
IMG_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 0
NUM_EPOCHS = 20
LR = 1e-3                # higher than B's 1e-4 because LoRA params are random init like a head
WEIGHT_DECAY = 1e-4
PATIENCE = 5

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# the 4 ranks we sweep
RANKS = [4, 8, 16, 32]

print(f'results dir: {RESULTS_DIR}')
print(f'ranks to test: {RANKS}')

results dir: /content/drive/MyDrive/Colab Notebooks/AdvancedDL/results/exp2_lora
ranks to test: [4, 8, 16, 32]


## 3. Copy dataset to local SSD + load sample index

Same pipeline as last exercise

In [ ]:
import shutil
# 8min L4
# 4min A100


LOCAL_ROOT = Path('/content/dataset_local')
DRIVE_RAW = DRIVE_ROOT / 'raw'
EXPECTED = {'train': 4654, 'val': 1125, 'test': 1124}

def copy_split(split):
    src = DRIVE_RAW / split / 'images'
    dst = LOCAL_ROOT / split / 'images'
    dst.mkdir(parents=True, exist_ok=True)
    if len(list(dst.glob('*.png'))) == EXPECTED[split]:
        print(f'  {split}: already complete')
        return
    print(f'  {split}: copying {EXPECTED[split]} images...')
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'  {split}: copied in {time.time()-t0:.0f}s')

print('copying dataset to local SSD...')
for split in ['train', 'val', 'test']:
    copy_split(split)
print('done.')

In [5]:
with open(INDEX_DIR / 'samples_train.json') as f: samples_train = json.load(f)
with open(INDEX_DIR / 'samples_val.json') as f:   samples_val = json.load(f)
with open(INDEX_DIR / 'samples_test.json') as f:  samples_test = json.load(f)
with open(INDEX_DIR / 'class_vocab.json') as f:   vocab = json.load(f)

CLASS_TO_IDX = vocab['class_to_idx']
IDX_TO_CLASS = {int(k): v for k, v in vocab['idx_to_class'].items()}
NUM_CLASSES = len(CLASS_TO_IDX)
CLASSES = vocab['keep_classes']

# rewrite drive paths to local ssd
for split_samples in [samples_train, samples_val, samples_test]:
    for s in split_samples:
        s['img_path'] = s['img_path'].replace(str(DRIVE_RAW), str(LOCAL_ROOT))

print(f'classes: {NUM_CLASSES}')
print(f'train: {len(samples_train):,}  val: {len(samples_val):,}  test: {len(samples_test):,}')

classes: 11
train: 11,000  val: 2,750  test: 2,750


## 4. Pre-load to RAM

If `preloaded_*.pt` exists in `/content/dataset_local/` from earlier sessions, this loads should be around 10s. If not it creates a new session in around 3 min. In case is ran from the beggining


In [ ]:
def preload_to_ram(samples):
    n = len(samples)
    imgs = torch.empty((n, 3, IMG_SIZE, IMG_SIZE), dtype=torch.uint8)
    labels = torch.empty((n,), dtype=torch.long)
    t0 = time.time()
    for i, s in enumerate(samples):
        img = Image.open(s['img_path']).convert('RGB')
        x1, y1, x2, y2 = s['bbox']
        crop = img.crop((x1, y1, x2, y2)).resize((IMG_SIZE, IMG_SIZE))
        imgs[i] = torch.from_numpy(np.asarray(crop)).permute(2, 0, 1)
        labels[i] = s['label']
        if (i+1) % 2000 == 0:
            print(f'  {i+1:,}/{n:,}')
    print(f'done: {n:,} in {time.time()-t0:.0f}s')
    return imgs, labels

class InMemoryDataset(Dataset):
    def __init__(self, imgs, labels):
        self.imgs = imgs; self.labels = labels
        self.mean = torch.tensor(IMAGENET_MEAN).view(3,1,1) * 255
        self.std = torch.tensor(IMAGENET_STD).view(3,1,1) * 255
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        x = self.imgs[i].float()
        return (x - self.mean) / self.std, self.labels[i]

CACHE_DIR = Path('/content/dataset_local')
def preload_or_load(samples, name):
    path = CACHE_DIR / f'preloaded_{name}.pt'
    if path.exists():
        print(f'{name}: loading cache...')
        cache = torch.load(path)
        return cache['imgs'], cache['labels']
    print(f'{name}: preloading...')
    imgs, labels = preload_to_ram(samples)
    torch.save({'imgs': imgs, 'labels': labels}, path)
    return imgs, labels

train_imgs, train_labels = preload_or_load(samples_train, 'train')
val_imgs, val_labels = preload_or_load(samples_val, 'val')
test_imgs, test_labels = preload_or_load(samples_test, 'test')

train_loader = DataLoader(InMemoryDataset(train_imgs, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(InMemoryDataset(val_imgs, val_labels),     batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(InMemoryDataset(test_imgs, test_labels),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'loaders ready')

train: preloading...


/tmp/ipykernel_5180/987304223.py:10: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  imgs[i] = torch.from_numpy(np.asarray(crop)).permute(2, 0, 1)


  2,000/11,000
  4,000/11,000
  6,000/11,000
  8,000/11,000
  10,000/11,000
